In [1]:
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C, WhiteKernel
from scipy.stats import norm

In [2]:


# ---------------------------
# Load data (4D inputs)
# ---------------------------
X = np.load("initial_inputs.npy")     # shape (N, 4)
y = np.load("initial_outputs.npy")    # shape (N,)

# Ensure correct shapes
X = X.reshape(-1, 4)  
y = y.reshape(-1)

# ---------------------------
# Fit Gaussian Process model
# ---------------------------
kernel = C(1.0, (1e-3, 1e3)) * RBF(length_scale=np.ones(4), length_scale_bounds=(1e-3, 1e3)) \
         + WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-8, 1e-3))

gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True)
gp.fit(X, y)

# ---------------------------
# Expected Improvement (EI)
# ---------------------------
def expected_improvement(X_cand, gp, y_best):
    mu, sigma = gp.predict(X_cand, return_std=True)
    sigma = sigma.reshape(-1, 1)

    # Avoid division by zero
    sigma = np.maximum(sigma, 1e-9)

    Z = (y_best - mu) / sigma
    ei = (y_best - mu) * norm.cdf(Z) + sigma * norm.pdf(Z)
    return ei.flatten()

# ---------------------------
# Generate random candidate points in 4D
# ---------------------------
lower = X.min(axis=0)
upper = X.max(axis=0)

n_candidates = 5000
X_cand = np.random.uniform(lower, upper, size=(n_candidates, 4))

# ---------------------------
# Compute EI and select next point
# ---------------------------
EI = expected_improvement(X_cand, gp, np.min(y))
next_point = X_cand[np.argmax(EI)]

print("Next proposed query point (4D):", next_point)


/Users/stephensefa/anaconda3/lib/python3.10/site-packages/sklearn/gaussian_process/kernels.py:420: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified lower bound 0.001. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/stephensefa/anaconda3/lib/python3.10/site-packages/sklearn/gaussian_process/kernels.py:420: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified lower bound 0.001. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/stephensefa/anaconda3/lib/python3.10/site-packages/sklearn/gaussian_process/kernels.py:420: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified lower bound 0.001. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/stephensefa/anaconda3/li

Next proposed query point (4D): [0.60759301 0.62528669 0.80606866 0.85092576]
